# NaverCafe NLP Pipeline (English Version)
Crawls community posts mentioning diaper brands, classifies by brand,
detects leakage/risk signals, and uploads results to data warehouse.

> **Note:** Original pipeline processes Korean community data.
> This version uses English-transliterated keywords for portfolio readability.

**Pipeline steps:**
1. Fetch recent community posts
2. Brand detection via keyword matching
3. Feature engineering: leakage flag, media flag, risk flag, risk keyword extraction
4. Deduplication against existing records
5. Upload to data warehouse

In [ ]:
# pip install snowflake-connector-python snowflake-sqlalchemy==1.5.1

In [ ]:
import os
import re
import time
import pandas as pd
import datetime
import openai
import snowflake.connector
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# ── BRAND KEYWORD DICTIONARY ──────────────────────────────────────────────
# Maps each brand to known product line names and aliases (English)
# Note: Original pipeline ran on Korean community data;
# this version uses English-transliterated equivalents for readability.

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['Huggies', 'MaxDry', 'MagicDry', 'NatureMade',
                           'MagicComfort', 'NatureMadeOrganic', 'NatureSummer']),
    'Pampers':  '|'.join(['Pampers', 'Armoni', 'BabyDry', 'TouchOfNature',
                           'AirChaCha', 'NightPants']),
    'Penelope': '|'.join(['Penelope', 'MiracleAllDay', 'ThinLight']),
    'Mamipoko': '|'.join(['Mamipoko', 'AirFit', 'GoldBreathable', 'LeafGanic']),
    'Bosomi':   '|'.join(['Bosomi', 'WonderByWonder', 'MegaDry', 'RealCottonOrganic']),
    'Kindoh':   '|'.join(['Kindoh', 'UpAndPlay', 'AllDay', 'SlimFit'])
}

TARGET_BRANDS = list(BRAND_KEYWORDS.keys())

# ── LEAKAGE DETECTION KEYWORDS ────────────────────────────────────────────
# Detects posts describing diaper leakage incidents

LEAKAGE_KEYWORDS = '|'.join([
    'leaking', 'leaked', 'leaks',
    'soaked', 'soaking', 'wet',
    'dripping', 'dripped',
    'overflow', 'overflowed',
    'backflow', 'drenched',
    'soggy', 'flooded', 'seeping'
])

# ── RISK / CONTAMINATION KEYWORDS ─────────────────────────────────────────
# Detects posts mentioning safety hazards or product contamination

RISK_KEYWORDS = [
    'foreign object', 'contamination', 'rust', 'rust water',
    'adhesive', 'mold', 'mould', 'insect', 'bug', 'fly',
    'metal particle', 'metal fragment',
    'hazardous substance', 'carcinogen', 'toxic',
    'VOC', 'benzene', 'toluene', 'styrene', 'xylene', 'TVOC'
]


In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set the following environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE

now = datetime.now() + timedelta(hours=9)  # KST
five_days_ago = (now - timedelta(days=5)).strftime('%Y-%m-%d')

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── DATA EXTRACTION ──────────────────────────────────────────────────────
# Fetch posts from the past 5 days, excluding deal/promotional boards

content_query = f"""
SELECT SEQ, POST_NUMBER, BOARD_PATH, CAFE_NAME, TITLE, CONTENTS,
       MEDIA, AUTHOR, URL, WDATE, RDATE, VIEWS, COMMENT_CNT, LIKES
FROM COMMUNITY_POSTS_VIEW
WHERE RDATE >= '{five_days_ago}'
AND NOT (
   BOARD_PATH IN ('Deal Board')
   OR BOARD_PATH LIKE '%HotDeal%'
   OR TITLE LIKE '%postpartum care%'
   OR TITLE LIKE '%baby products%'
   OR TITLE LIKE '%review%'
   OR TITLE LIKE '%hot deal%'
);
"""

content = pd.DataFrame(cur.execute(content_query))
content_columns = ['SEQ', 'POST_NUMBER', 'BOARD_PATH', 'CAFE_NAME', 'TITLE',
                   'CONTENTS', 'MEDIA', 'AUTHOR', 'URL', 'WDATE', 'RDATE',
                   'VIEWS', 'COMMENT_CNT', 'LIKES']
content.columns = content_columns
content['CONTENTS'] = content['CONTENTS'].fillna('')
content['RDATE'] = pd.to_datetime(content['RDATE']).dt.strftime('%Y-%m-%d')
content['WDATE'] = pd.to_datetime(content['WDATE']).dt.strftime('%Y-%m-%d')
content['YEAR'] = pd.to_datetime(content['WDATE']).dt.year
content['MONTH'] = pd.to_datetime(content['WDATE']).dt.month

In [ ]:
# ── BRAND DETECTION ──────────────────────────────────────────────────────

def detect_brands(content_text, title):
    """Return list of brands mentioned in the content or title."""
    return [
        brand for brand, pattern in BRAND_KEYWORDS.items()
        if re.search(pattern, content_text, re.IGNORECASE)
        or re.search(pattern, title, re.IGNORECASE)
    ]

# Expand rows: one row per brand mention per post
expanded_rows = []
for _, row in content.iterrows():
    for brand in detect_brands(row['CONTENTS'], row['TITLE']):
        expanded_row = row.copy()
        expanded_row['BRAND'] = brand
        expanded_rows.append(expanded_row)

expanded_content = pd.DataFrame(expanded_rows)
expanded_content['UNIQUE_ID'] = expanded_content.apply(
    lambda row: f"{row['SEQ']}-{row['BRAND']}", axis=1
)
expanded_content = expanded_content[expanded_content['BRAND'].isin(TARGET_BRANDS)]

In [ ]:
# ── FEATURE ENGINEERING ─────────────────────────────────────────────────

def check_leakage(row):
    """Return 1 if post mentions leakage, else 0."""
    return int(bool(
        re.search(LEAKAGE_KEYWORDS, row['CONTENTS'], re.IGNORECASE) or
        re.search(LEAKAGE_KEYWORDS, row['TITLE'], re.IGNORECASE)
    ))

def find_risk_keywords(content_text, title):
    """Return comma-separated string of detected risk keywords."""
    found = {kw for kw in RISK_KEYWORDS
             if re.search(kw, content_text, re.IGNORECASE)
             or re.search(kw, title, re.IGNORECASE)}
    return ', '.join(found)

def find_nearby_sentences(text, keywords, window=2):
    """Extract sentences surrounding a keyword match (context window)."""
    sentences = re.split(r'(?<=[.!?]) +', text)
    indices = [i for i, s in enumerate(sentences)
               if any(re.search(kw, s, re.IGNORECASE) for kw in keywords)]
    nearby = set()
    for idx in indices:
        for i in range(max(0, idx - window), min(len(sentences), idx + window + 1)):
            nearby.add(sentences[i])
    return ' '.join(nearby)

def check_risk(row):
    """Return 1 if risk keyword appears near a brand mention, else 0."""
    nearby_text = ''
    for brand, pattern in BRAND_KEYWORDS.items():
        if re.search(pattern, row['CONTENTS'], re.IGNORECASE):
            nearby_text += ' ' + find_nearby_sentences(row['CONTENTS'], [pattern])
        if re.search(pattern, row['TITLE'], re.IGNORECASE):
            nearby_text += ' ' + find_nearby_sentences(row['TITLE'], [pattern])
    return int(any(re.search(rk, nearby_text, re.IGNORECASE) for rk in RISK_KEYWORDS))

brand_keywords_list = {brand: pattern.split('|') for brand, pattern in BRAND_KEYWORDS.items()}

def extract_brand_sentence(row):
    """Extract the sentence most relevant to the detected brand."""
    keywords_for_brand = brand_keywords_list.get(row['BRAND'], [])
    sentences = re.split(r'[;.!?\n]', row['CONTENTS'])
    for i, s in enumerate(sentences):
        if any(kw.lower() in s.lower() for kw in keywords_for_brand):
            prev_s = sentences[i-1] if i > 0 else ''
            next_s = sentences[i+1] if i < len(sentences)-1 else ''
            return ' '.join(filter(None, [prev_s, s, next_s])).strip()
    return row['TITLE']

expanded_content['LEAKAGE_YN'] = expanded_content.apply(check_leakage, axis=1)
expanded_content['MEDIA_YN'] = expanded_content['MEDIA'].apply(
    lambda x: 1 if pd.notnull(x) and x != '' else 0
)
expanded_content['RISK_YN'] = expanded_content.apply(check_risk, axis=1)
expanded_content['RISK_KEYWORD'] = expanded_content.apply(
    lambda row: find_risk_keywords(row['CONTENTS'], row['TITLE']), axis=1
)
expanded_content['CONTENTS_S'] = expanded_content.apply(extract_brand_sentence, axis=1)

In [ ]:
# ── DEDUPLICATION ────────────────────────────────────────────────────────
# Load existing records and keep the latest version of each post-brand pair

orig_columns = ['UNIQUE_ID', 'SEQ', 'POST_NUMBER', 'BOARD_PATH', 'CAFE_NAME',
                'TITLE', 'CONTENTS', 'MEDIA', 'AUTHOR', 'URL', 'WDATE', 'RDATE',
                'VIEWS', 'COMMENT_CNT', 'LIKES', 'YEAR', 'MONTH', 'BRAND',
                'LEAKAGE_YN', 'MEDIA_YN', 'RISK_YN', 'RISK_KEYWORD', 'CONTENTS_S']

orig_content = pd.DataFrame(cur.execute('SELECT * FROM NLP_TABLE').fetchall(), columns=orig_columns)

combined = pd.concat(
    [orig_content.set_index(['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND']),
     expanded_content.set_index(['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND'])],
    axis=0
).reset_index()

updated_content = combined.drop_duplicates(
    subset=['POST_NUMBER', 'TITLE', 'CONTENTS', 'BRAND'], keep='last'
)

In [ ]:
# ── UPLOAD TO DATA WAREHOUSE ─────────────────────────────────────────────

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    con.execute('DELETE FROM NLP_TABLE')
    updated_content.to_sql(name='NLP_TABLE', con=con, if_exists='append',
                           method=pd_writer, index=False)

print(f'Uploaded {len(updated_content)} records')